In [ ]:
## 1. Configuração

import requests
import json
import time
import pandas as pd

In [ ]:
## 2. Exploração da API

# Escolhe qualquer username público do Lichess
USERNAME = "DrNykterstein"  # Magnus Carlsen no Lichess
url = f"https://lichess.org/api/games/user/{USERNAME}"

# Requisição com análise completa do Stockfish
params = {
    "rated":    "true",
    "max":      1,
    "opening":  "true",
    "evals":    "true",    # avaliações do Stockfish por lance
    "clocks":   "true",    # tempo no relógio por lance
    "analysed": "true",    # só partidas já analisadas
}
headers = {"Accept": "application/x-ndjson"}

response = requests.get(url, params=params, headers=headers)
game = json.loads(response.text)

# Inspeciona estrutura
print("CLOCKS:", game["clocks"][:5])
print("ANALYSIS:", game["analysis"][:5])

# Lances com julgamento do Stockfish
erros_exemplo = [l for l in game["analysis"] if "judgment" in l]
print(f"\nLances com erro: {len(erros_exemplo)}")
print(erros_exemplo[0])

Status: 200
{"id":"kAdOQKeh","rated":true,"variant":"standard","speed":"blitz","perf":"blitz","createdAt":1775677143033,"lastMoveAt":1775677513708,"status":"resign","source":"arena","players":{"white":{"user":{"name":"respects_55","id":"respects_55"},"rating":2644,"ratingDiff":-5,"provisional":true},"black":{"user":{"name":"DrNykterstein","title":"GM","flair":"people.santa-claus-light-skin-tone","patron":true,"patronColor":10,"id":"drnykterstein"},"rating":3145,"ratingDiff":8,"provisional":true}},"winner":"black","opening":{"eco":"B02","name":"Alekhine Defense: Sämisch Attack","ply":5},"moves":"e4 Nf6 e5 Nd5 Nc3 Nxc3 dxc3 d6 Nf3 Nc6 Bb5 a6 Bxc6+ bxc6 O-O f6 exf6 exf6 Nd4 Qd7 Qh5+ g6 Qf3 Kf7 Nxc6 Bb7 Nd8+ Rxd8 Qxb7 Qb5 Qxc7+ Rd7 c4 Rxc7 cxb5 axb5 c3 Be7 Rd1 Ra8 a3 f5 Be3 Ke6 Rd3 Bf6 Rad1 Rc6 Bf4 Be5 Re1 Kf6 Bxe5+ dxe5 Rd5 e4 Rxb5 Rd6 g3 Rd2 c4 Rc2 c5 Rd8 Rb6+ Kg5 Rd6 Ra8 c6 Rxb2 Rc1 Rc8 c7 Rb7 Rdc6 Ra7 R1c3 f4 R6c5+ Kg4 gxf4 Kxf4 Kg2 g5 R5c4 Ke5 Rc5+ Kf4 f3 exf3+ Kf2 Ke4 R5c4+ Kf5 Kxf3 

In [ ]:
## 3. Extração de Erros

def extrair_erros(game):
    erros = []
    
    analise  = game.get("analysis", [])
    relogios = game.get("clocks", [])
    
    speed        = game.get("speed", "unknown")
    abertura     = game.get("opening", {}).get("name", "Desconhecida")
    rating_white = game["players"]["white"].get("rating")
    rating_black = game["players"]["black"].get("rating")
    
    for i, lance in enumerate(analise):
        if "judgment" not in lance:
            continue  # lance sem erro, pula
        
        cor           = "white" if i % 2 == 0 else "black"
        numero_lance  = (i // 2) + 1
        rating_jogador = rating_white if cor == "white" else rating_black
        relogio        = relogios[i] / 100 if i < len(relogios) else None  # converte para segundos
        
        # Fase do jogo
        if numero_lance <= 15:
            fase = "Abertura"
        elif numero_lance <= 35:
            fase = "Meio-jogo"
        else:
            fase = "Final"
        
        erros.append({
            "game_id":        game["id"],
            "speed":          speed,
            "numero_lance":   numero_lance,
            "fase":           fase,
            "cor":            cor,
            "rating_jogador": rating_jogador,
            "relogio_seg":    relogio,
            "tipo_erro":      lance["judgment"]["name"],  # Inaccuracy / Mistake / Blunder
            "eval":           lance.get("eval"),
            "abertura":       abertura,
        })
    
    return erros

# Testa com a partida que já temos
resultado = extrair_erros(game)
df_teste = pd.DataFrame(resultado)
print(df_teste)

     game_id  speed  numero_lance       fase    cor  rating_jogador  \
0   kAdOQKeh  blitz             8   Abertura  black            3145   
1   kAdOQKeh  blitz            12   Abertura  black            3145   
2   kAdOQKeh  blitz            29  Meio-jogo  white            2644   
3   kAdOQKeh  blitz            32  Meio-jogo  white            2644   
4   kAdOQKeh  blitz            33  Meio-jogo  black            3145   
5   kAdOQKeh  blitz            35  Meio-jogo  black            3145   
6   kAdOQKeh  blitz            47      Final  white            2644   
7   kAdOQKeh  blitz            47      Final  black            3145   
8   kAdOQKeh  blitz            49      Final  white            2644   
9   kAdOQKeh  blitz            51      Final  white            2644   
10  kAdOQKeh  blitz            51      Final  black            3145   
11  kAdOQKeh  blitz            52      Final  white            2644   
12  kAdOQKeh  blitz            58      Final  white            2644   
13  kA

In [ ]:
## 4. Coleta em Escala

def buscar_usuarios_por_faixa():
    """
    Snowball sampling: começa com seeds conhecidos de cada faixa
    e expande pelos oponentes das partidas deles.
    O matchmaking do Lichess garante que oponentes têm rating similar.
    """

    # Seeds verificados manualmente no Lichess — um por faixa
    # Troca se algum não existir mais
    seeds_por_faixa = {
        "Iniciante (400-1000)":      ["acgusta",       "chess_enjoyer01", "beginner_moves"],
        "Intermediário (1000-1600)": ["Dharmes2620009", "HEVSEL",          "chards01"],
        "Avançado (1600-2200)":      ["staxmann",       "wonderland305",   "noprepleak"],
        "Expert (2200+)":            ["DrNykterstein",  "nihalsarin2002",  "RebeccaHarris"],
    }

    resultado = {faixa: [] for faixa in seeds_por_faixa}

    for faixa, seeds in seeds_por_faixa.items():
        print(f"\nColetando: {faixa}")
        usernames = set()

        for seed in seeds:
            print(f"  seed: {seed}")
            try:
                url = f"https://lichess.org/api/games/user/{seed}"
                params = {
                    "max":      50,       # 50 partidas por seed
                    "rated":    "true",
                    "analysed": "true",   # só partidas com análise
                }
                r = requests.get(
                    url,
                    params=params,
                    headers={"Accept": "application/x-ndjson"},
                    timeout=15
                )

                for line in r.text.strip().split("\n"):
                    if not line:
                        continue
                    try:
                        g = json.loads(line)
                        # Extrai os dois jogadores de cada partida
                        w = g["players"]["white"].get("user", {}).get("name")
                        b = g["players"]["black"].get("user", {}).get("name")
                        if w:
                            usernames.add(w)
                        if b:
                            usernames.add(b)
                    except:
                        continue

                time.sleep(1.5)  # respeita rate limit

            except Exception as e:
                print(f"    erro: {e}")
                continue

        resultado[faixa] = list(usernames)
        print(f"  → {len(resultado[faixa])} usuários coletados")

    return resultado


usuarios_por_faixa = buscar_usuarios_por_faixa()

print("\n── Resumo ──────────────────────")
for faixa, users in usuarios_por_faixa.items():
    print(f"  {faixa}: {len(users)} usuários → {users[:3]}")


def coletar_partidas_e_erros(usuarios_por_faixa, partidas_por_usuario=20):
    """
    Para cada usuário de cada faixa:
      1. Busca partidas analisadas (todos os modos)
      2. Extrai os erros usando a função extrair_erros() que já criamos
    """
    todos_erros = []
    
    for faixa, usernames in usuarios_por_faixa.items():
        print(f"\nColetando partidas: {faixa}")
        erros_faixa = 0
        
        for username in usernames:
            try:
                url = f"https://lichess.org/api/games/user/{username}"
                params = {
                    "max":      partidas_por_usuario,
                    "rated":    "true",
                    "analysed": "true",  # só partidas com análise do Stockfish
                    "evals":    "true",
                    "clocks":   "true",
                    "opening":  "true",
                }
                r = requests.get(
                    url,
                    params=params,
                    headers={"Accept": "application/x-ndjson"},
                    timeout=15
                )
                
                for line in r.text.strip().split("\n"):
                    if not line:
                        continue
                    try:
                        game = json.loads(line)
                        erros = extrair_erros(game)
                        
                        # Adiciona a faixa do jogador seed em cada erro
                        for e in erros:
                            e["faixa_seed"] = faixa
                        
                        todos_erros.extend(erros)
                        erros_faixa += len(erros)
                    except:
                        continue
                
                time.sleep(1.5)  # respeita rate limit
                
            except Exception as ex:
                print(f"  erro em {username}: {ex}")
                continue
        
        print(f"  → {erros_faixa} erros extraídos")
    
    return pd.DataFrame(todos_erros)


print("Iniciando coleta... (pode levar alguns minutos)")
df = coletar_partidas_e_erros(usuarios_por_faixa)

print(f"\n── Resultado final ──────────────────")
print(f"  Total de erros: {len(df):,}")
print(f"\nDistribuição por faixa:")
print(df["faixa_seed"].value_counts())
print(f"\nDistribuição por modalidade:")
print(df["speed"].value_counts())
print(f"\nDistribuição por tipo de erro:")
print(df["tipo_erro"].value_counts())


def coletar_usuarios_aleatorios(n_usuarios=500):
    """
    Coleta usuários aleatórios via torneios abertos do Lichess.
    Sem filtro de rating — reflete a distribuição real da plataforma.
    """
    usernames = set()

    print("Buscando torneios abertos...")
    url = "https://lichess.org/api/tournament"
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=10)
    data = r.json()

    torneios = (
        data.get("created", []) +
        data.get("started", []) +
        data.get("finished", [])
    )

    # Filtra só torneios sem restrição de rating
    torneios_abertos = [
        t for t in torneios
        if not t.get("conditions", {}).get("maxRating")
        and not t.get("conditions", {}).get("minRating")
    ]
    print(f"  {len(torneios_abertos)} torneios abertos encontrados")

    for torneio in torneios_abertos:
        if len(usernames) >= n_usuarios:
            break

        tid = torneio.get("id")
        try:
            url2 = f"https://lichess.org/api/tournament/{tid}/results"
            r2 = requests.get(
                url2,
                headers={"Accept": "application/x-ndjson"},
                timeout=10
            )
            for line in r2.text.strip().split("\n"):
                if not line:
                    continue
                d = json.loads(line)
                if "username" in d:
                    usernames.add(d["username"])

            print(f"  torneio {tid}: {len(usernames)} usuários acumulados")
            time.sleep(0.8)

        except:
            continue

    return list(usernames)[:n_usuarios]


usuarios_aleatorios = coletar_usuarios_aleatorios(n_usuarios=500)
print(f"\nTotal coletado: {len(usuarios_aleatorios)} usuários")
print("Exemplos:", usuarios_aleatorios[:5])


Coletando: Iniciante (400-1000)
  seed: acgusta
  seed: chess_enjoyer01
  seed: beginner_moves
  → 51 usuários coletados

Coletando: Intermediário (1000-1600)
  seed: Dharmes2620009


KeyboardInterrupt: 

In [20]:
## 5. Tratamento e Validação

def classificar_faixa(rating):
    if rating is None:
        return None
    elif rating < 1000:
        return "Iniciante (400-1000)"
    elif rating < 1600:
        return "Intermediário (1000-1600)"
    elif rating < 2200:
        return "Avançado (1600-2200)"
    else:
        return "Expert (2200+)"

# Substitui faixa_seed pela faixa real do jogador
df["faixa"] = df["rating_jogador"].apply(classificar_faixa)

# Remove linhas sem rating
df = df[df["faixa"].notna()].copy()

print("Distribuição por faixa (corrigida):")
print(df["faixa"].value_counts())

print("\nRatings por faixa (corrigido):")
print(df.groupby("faixa")["rating_jogador"].describe()[["min","mean","max"]])


# Usuários confirmados nessas faixas que já encontramos antes
seeds_extras = {
    "Iniciante (400-1000)": [
        "Malacothius", "acgusta", "chess_enjoyer01"
    ],
    "Intermediário (1000-1600)": [
        "mgdk", "chards01", "VitinhoAgiota",
        "yousefsadi99", "iing1967", "DanYack"
    ]
}

df_extra = coletar_partidas_e_erros(
    seeds_extras,
    partidas_por_usuario=50   # mais partidas por usuário dessa vez
)

# Corrige faixa pelo rating real
df_extra["faixa"] = df_extra["rating_jogador"].apply(classificar_faixa)
df_extra = df_extra[df_extra["faixa"].notna()].copy()

# Junta com o dataframe principal
df = pd.concat([df, df_extra], ignore_index=True)

# Remove duplicatas (mesma partida pode aparecer dos dois lados)
df = df.drop_duplicates(subset=["game_id", "numero_lance", "cor"])

print("Distribuição atualizada:")
print(df["faixa"].value_counts())


# Coleta partidas dos usuários aleatórios
print("Coletando partidas dos usuários aleatórios...")
df_aleatorio = coletar_partidas_e_erros(
    {"Amostra Aleatória": usuarios_aleatorios},
    partidas_por_usuario=20
)

# Classifica por rating real (substitui faixa_seed)
df_aleatorio["faixa"] = df_aleatorio["rating_jogador"].apply(classificar_faixa)
df_aleatorio = df_aleatorio[df_aleatorio["faixa"].notna()].copy()

print("\nDistribuição da amostra aleatória:")
print(df_aleatorio["faixa"].value_counts())

# Junta com o dataset que já tínhamos
df_final = pd.concat([df, df_aleatorio], ignore_index=True)
df_final = df_final.drop_duplicates(subset=["game_id", "numero_lance", "cor"])

print("\n── Distribuição final ───────────────────────────")
print(df_final["faixa"].value_counts())
print(f"\nTotal de erros: {len(df_final):,}")

Distribuição por faixa (corrigida):
faixa
Avançado (1600-2200)         60937
Expert (2200+)               23659
Intermediário (1000-1600)     8065
Iniciante (400-1000)           187
Name: count, dtype: int64

Ratings por faixa (corrigido):
                              min         mean     max
faixa                                                 
Avançado (1600-2200)       1600.0  1844.134483  2199.0
Expert (2200+)             2200.0  2611.020965  3407.0
Iniciante (400-1000)        772.0   936.866310   999.0
Intermediário (1000-1600)  1000.0  1475.644141  1599.0

Coletando partidas: Iniciante (400-1000)


KeyboardInterrupt: 

In [ ]:
## 6. Exportação

output_path = "../data/blunders.csv"
df_final.to_csv(output_path, index=False)

print(f"✅ CSV exportado: {output_path}")
print(f"   Linhas: {len(df_final):,}")
print(f"   Colunas: {list(df_final.columns)}")
print(f"\nPrimeiras linhas:")
df_final.head()

✅ CSV exportado: ../data/blunders.csv
   Linhas: 228,999
   Colunas: ['game_id', 'speed', 'numero_lance', 'fase', 'cor', 'rating_jogador', 'relogio_seg', 'tipo_erro', 'eval', 'abertura', 'faixa_seed', 'faixa']

Primeiras linhas:


,game_id,speed,numero_lance,fase,cor,rating_jogador,relogio_seg,tipo_erro,eval,abertura,faixa_seed,faixa
0,8j6WtgMF,blitz,4,Abertura,black,1800,294.03,Inaccuracy,80.0,"Queen's Pawn Game: Accelerated London System, ...",Iniciante (400-1000),Avançado (1600-2200)
1,8j6WtgMF,blitz,5,Abertura,black,1800,292.19,Inaccuracy,94.0,"Queen's Pawn Game: Accelerated London System, ...",Iniciante (400-1000),Avançado (1600-2200)
2,8j6WtgMF,blitz,8,Abertura,white,1821,276.51,Inaccuracy,18.0,"Queen's Pawn Game: Accelerated London System, ...",Iniciante (400-1000),Avançado (1600-2200)
3,8j6WtgMF,blitz,10,Abertura,white,1821,270.91,Inaccuracy,-13.0,"Queen's Pawn Game: Accelerated London System, ...",Iniciante (400-1000),Avançado (1600-2200)
4,8j6WtgMF,blitz,14,Abertura,white,1821,263.15,Blunder,-186.0,"Queen's Pawn Game: Accelerated London System, ...",Iniciante (400-1000),Avançado (1600-2200)
